# 🛡️ Phish Shield - LangGraph agent (Colab)
LangGraph state machine: extract → links → language → score → judge (loops if unsure) → action / clear.
**Setup (optional but recommended):** 🔑 *Secrets* → add `GEMINI_API_KEY`. Without it the agent runs in offline rule-engine mode.

In [ ]:
!pip install -q "google-genai>=2.3.0" langgraph

### Get the code (run ONE option)
**Option A** - after you push to GitHub (edit your username):

In [ ]:
GITHUB_USER = "YOUR_GITHUB_USERNAME"
import os
if not os.path.exists("phish-shield-agent"):
    !git clone https://github.com/{GITHUB_USER}/phish-shield-agent.git
%cd phish-shield-agent

**Option B** - not pushed yet? Upload the repo zip:

In [ ]:
from google.colab import files
up = files.upload()                      # choose phish-shield-agent.zip
!unzip -oq phish-shield-agent.zip
%cd phish-shield-agent

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    print("No key found -> offline rule-engine mode")
from shield import analyze, build_graph

### Graph structure

In [ ]:
g = build_graph()
print(g.get_graph().draw_mermaid())
try:
    from IPython.display import Image, display
    display(Image(g.get_graph().draw_mermaid_png()))
except Exception as e:
    print("PNG preview unavailable:", e)

### Run on the sample messages

In [ ]:
from pathlib import Path
for p in sorted(Path("sample_emails").glob("*.txt")):
    r = analyze(p.read_text())
    print(f"\n=== {p.name} ===")
    print("VERDICT:", r["verdict"], "| score:", r["rule_score"], "| confidence:", r["confidence"])
    print("WHY:", r["reasoning"])
    print("TRACE:", *r["trace"], sep="\n  ")
    print("PLAN:\n" + r["action_plan"])

### Try your own message

In [ ]:
msg = input("Paste a suspicious message: ")
r = analyze(msg)
print(r["verdict"], r["rule_score"], "\n", r["reasoning"], "\n", r["action_plan"])

### Unit tests

In [ ]:
!pip install -q pytest
!python -m pytest -q tests